# LightGBM

A gradient-boosted tree classifier compares `application_only` and `application_history` features from the shared preparation cache. Both variants use the same applicant splits.

Category vocabularies are fitted on training rows only. Validation ROC-AUC selects the model; classification metrics use a fixed 0.5 threshold. Holdout and competition data are not evaluated.


In [1]:

%pip install -q pandas numpy "scikit-learn>=1.3" lightgbm

from pathlib import Path
import sys

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts" / "project_data.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Notebook working directory must be inside this repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import gc
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display
from scripts.project_data import FEATURE_SETS, load_prepared_data, get_partitions
from scripts.experiment_utils import (
    validation_metrics, split_fingerprint, feature_schema_fingerprint,
    package_versions, save_run_record,
)


FEATURE_SET = "application_history"  # Alternative: "application_only"
SPLIT_STRATEGY = "stratified"      # Alternative: "random"
RANDOM_STATE = 42
CLASSIFICATION_THRESHOLD = 0.5
ACTIVE_PARTITIONS = ("train", "validation")

## 1. Feature set and shared splits

The prepared cache contains application-only and application-plus-history features with identical applicant assignments. This notebook selects training and validation rows for `FEATURE_SET`, records the split fingerprint, and releases the full cache bundle.


In [2]:
preparation_started = perf_counter()
prepared_data = load_prepared_data()
run_split_fingerprint = split_fingerprint(
    prepared_data["split_ids"][SPLIT_STRATEGY], prepared_data["labels"],
)
data = get_partitions(
    prepared_data, FEATURE_SET, SPLIT_STRATEGY, partitions=ACTIVE_PARTITIONS,
)
raw_inputs = data["raw_inputs"]
labels = data["labels"]
applicant_ids = data["applicant_ids"]
audit_by_partition = data["audit_by_partition"]
numeric_features = data["numeric_features"]
categorical_features = data["categorical_features"]
run_feature_schema = feature_schema_fingerprint(raw_inputs["train"].columns)
prepared_schema_version = prepared_data["schema_version"]
raw_feature_count = raw_inputs["train"].shape[1]
run_row_counts = {part: len(raw_inputs[part]) for part in ACTIVE_PARTITIONS}
run_positive_rates = {part: float(labels[part].mean()) for part in ACTIVE_PARTITIONS}
del prepared_data, data
gc.collect()
assert set(raw_inputs) == {"train", "validation"}
display(pd.DataFrame([
    {"feature_set": FEATURE_SET, "partition": part, "rows": len(frame), "raw_features": frame.shape[1]}
    for part, frame in raw_inputs.items()
]))
print("Protocol:", SPLIT_STRATEGY)

,feature_set,partition,rows,raw_features
0,application_history,train,215257,303
1,application_history,validation,46127,303


Protocol: stratified


## 2. LightGBM preprocessing

Numeric features retain their original scales and missing values. Categorical vocabularies are learned from training rows and reused on validation; missing and unseen categories receive missing codes. No imputation or one-hot encoding is applied. [Native missing and categorical handling](https://lightgbm.readthedocs.io/en/stable/Advanced-Topics.html).


In [3]:
def prepare_lightgbm(frame, category_levels):
    prepared = frame.copy()
    for column, levels in category_levels.items():
        values = prepared[column].map(lambda value: str(value) if pd.notna(value) else np.nan)
        prepared[column] = pd.Categorical(values, categories=levels, ordered=False)
    return prepared


lgbm_category_levels = {
    column: raw_inputs["train"][column].dropna().astype(str).unique().tolist()
    for column in categorical_features
}
lgbm_cat_features = categorical_features
lgbm_inputs = {
    part: prepare_lightgbm(raw_inputs[part], lgbm_category_levels)
    for part in ACTIVE_PARTITIONS
}

## 3. Input alignment

The prepared DataFrames retain their applicant-ID index, and categories match the training vocabulary. Numeric NaNs are intentional. Raw frames are released before fitting.

In [4]:
for part in ACTIVE_PARTITIONS:
    frame = lgbm_inputs[part]
    raw = raw_inputs[part]
    assert frame.index.equals(raw.index) and frame.columns.equals(raw.columns)
    np.testing.assert_array_equal(frame.index.to_numpy(), applicant_ids[part])
    assert not np.isinf(frame[numeric_features].to_numpy()).any()
    pd.testing.assert_frame_equal(frame[numeric_features], raw[numeric_features])
    for column in lgbm_cat_features:
        assert frame[column].cat.categories.tolist() == lgbm_category_levels[column]
        assert not frame[column].cat.ordered
    assert frame.index.equals(labels[part].index)
display(pd.DataFrame([
    {"partition": part, "rows": len(lgbm_inputs[part]), "features": lgbm_inputs[part].shape[1]}
    for part in ACTIVE_PARTITIONS
]))
del raw_inputs, raw, frame
gc.collect()
preparation_seconds = perf_counter() - preparation_started
print(f"Preparation finished in {preparation_seconds:.1f}s. No training yet.")

,partition,rows,features
0,train,215257,303
1,validation,46127,303


Preparation finished in 10.6s. No training yet.


## 4. Model configuration

Binary gradient boosting uses 31 leaves, learning rate 0.05, at most 1,000 trees, at least 50 rows per leaf, 80% row/column sampling, and L2 regularization 1.

Validation ROC-AUC controls early stopping with patience of 75 rounds, and predictions use the best iteration. Computation uses four CPU threads and seed 42, without class weighting. [Classifier API](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMClassifier.html).


In [5]:
try:
    import lightgbm as lgb
except ImportError as exc:
    raise ImportError("Run %pip install lightgbm in this kernel, then restart/rerun.") from exc

CONFIGURATION_NAME = "starter_v1"
LGBM_CONFIG = {
    "objective": "binary", "boosting_type": "gbdt", "n_estimators": 1000,
    "learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 50,
    "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8,
    "reg_lambda": 1.0, "class_weight": None, "random_state": RANDOM_STATE,
    "n_jobs": 4, "metric": "auc", "verbosity": -1,
    "deterministic": True, "force_col_wise": True,
}
EARLY_STOPPING_ROUNDS = 75
print(LGBM_CONFIG)

{'objective': 'binary', 'boosting_type': 'gbdt', 'n_estimators': 1000, 'learning_rate': 0.05, 'num_leaves': 31, 'min_child_samples': 50, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'class_weight': None, 'random_state': 42, 'n_jobs': 4, 'metric': 'auc', 'verbosity': -1, 'deterministic': True, 'force_col_wise': True}


## 5. Training

The classifier fits training rows only. Validation ROC-AUC controls early stopping and selects the retained model; labeled holdout and competition rows are excluded.


In [6]:
model = lgb.LGBMClassifier(**LGBM_CONFIG)
training_started = perf_counter()
model.fit(
    lgbm_inputs["train"], labels["train"],
    eval_set=[(lgbm_inputs["validation"], labels["validation"])],
    eval_names=["validation"], eval_metric="auc",
    categorical_feature=lgbm_cat_features,
    callbacks=[
        lgb.early_stopping(EARLY_STOPPING_ROUNDS, first_metric_only=True),
        lgb.log_evaluation(100),
    ],
)
training_seconds = perf_counter() - training_started
best_iteration = int(model.best_iteration_ or model.n_estimators_)
prediction_started = perf_counter()
validation_probability = model.predict_proba(
    lgbm_inputs["validation"], num_iteration=best_iteration,
)[:, 1]
prediction_seconds = perf_counter() - prediction_started
run_parameters = {**LGBM_CONFIG, "early_stopping_rounds": EARLY_STOPPING_ROUNDS}
training_history = model.evals_result_
model_feature_count = lgbm_inputs["train"].shape[1]
print(f"Selected trees: {best_iteration}; training time: {training_seconds:.1f}s")

C:\Users\CRNeu\Documents\Data4401\Data4401-Project2\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 75 rounds


[100]	validation's auc: 0.777204


[200]	validation's auc: 0.783612


[300]	validation's auc: 0.784787


[400]	validation's auc: 0.785109


Early stopping, best iteration is:
[367]	validation's auc: 0.785255
Evaluated only: auc


Selected trees: 367; training time: 44.0s


## 6. Validation metrics

ROC-AUC is the primary metric, complemented by average precision, log loss, accuracy, precision, recall, and F1. Classification metrics use a fixed 0.5 threshold. The confusion matrix places actual labels in rows and predicted labels in columns.

A run record in `results/metrics/` stores metrics, configuration, timings, learning history, library versions, and split/feature fingerprints for cross-model comparison.


In [7]:
metrics = validation_metrics(
    labels["validation"], validation_probability, threshold=CLASSIFICATION_THRESHOLD,
)
display(pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in metrics.items() if key not in {"confusion_matrix", "threshold"}
]))
counts = metrics["confusion_matrix"]
display(pd.DataFrame(
    [[counts["tn"], counts["fp"]], [counts["fn"], counts["tp"]]],
    index=pd.Index(["0", "1"], name="Actual"),
    columns=pd.Index(["0", "1"], name="Predicted"),
))
run_record = {
    "model": "lightgbm", "feature_set": FEATURE_SET, "split_strategy": SPLIT_STRATEGY,
    "configuration": CONFIGURATION_NAME, "seed": RANDOM_STATE, "evaluation_partition": "validation",
    "optimizing_metric": "roc_auc", "split_fingerprint": run_split_fingerprint,
    "feature_schema_fingerprint": run_feature_schema, "prepared_schema_version": prepared_schema_version,
    "raw_feature_count": raw_feature_count, "model_feature_count": model_feature_count,
    "rows": run_row_counts, "positive_rates": run_positive_rates, "parameters": run_parameters,
    "selected_iteration": best_iteration, "preparation_seconds": preparation_seconds,
    "training_seconds": training_seconds, "prediction_seconds": prediction_seconds,
    "metrics": metrics, "training_history": training_history, "versions": package_versions("lightgbm"),
    "holdout_evaluated": False, "class_weighting": "none", "threshold_tuned": False,
}
result_path = save_run_record(run_record)
print(f"Saved validation results: {result_path}")
print("Holdout not evaluated. No Kaggle submission created.")

,metric,value
0,roc_auc,0.785255
1,average_precision,0.283053
2,accuracy,0.919895
3,f1,0.092806
4,precision,0.541547
5,recall,0.050752
6,log_loss,0.237498


Predicted,0,1
Actual,,
0,42243,160
1,3535,189


Saved validation results: C:\Users\CRNeu\Documents\Data4401\Data4401-Project2\results\metrics\lightgbm_application_history_stratified_starter_v1_20261006T080557325548Z_da5c5738.json
Holdout not evaluated. No Kaggle submission created.


## 7. Applicant-level validation predictions

The existing validation probabilities are saved to `results/predictions/` as a CSV with matching run metadata. Each row contains applicant ID, actual target, predicted probability and label, error type (TN/FP/FN/TP), gender, age, age group, and run ID.

Metadata links predictions to the metric record, split, configuration, and threshold. These artifacts support downstream error and fairness analysis without refitting. Only validation rows are exported; missing audit categories remain `UNKNOWN` and missing numeric ages remain blank.


In [8]:
from scripts import experiment_utils as _prediction_utils

PREDICTION_EXPORT_DIR = None  # Default: results/predictions/ in this repo.
prediction_csv_path, prediction_metadata_path = _prediction_utils.export_validation_predictions(
    applicant_ids=applicant_ids["validation"],
    labels=labels["validation"],
    probabilities=validation_probability,
    audit=audit_by_partition["validation"],
    result_path=result_path,
    directory=PREDICTION_EXPORT_DIR,
)
print(f"Validation predictions: {prediction_csv_path}")
print(f"Run metadata: {prediction_metadata_path}")
print(f"CSV: {prediction_csv_path.stat().st_size / 1024**2:.2f} MiB; metadata: {prediction_metadata_path.stat().st_size / 1024:.1f} KiB")
print(f"Rows: {len(labels['validation']):,}; feature set: {run_record['feature_set']}")
print("Validation-only predictions exported; no model refitted.")


Validation predictions: C:\Users\CRNeu\Documents\Data4401\Data4401-Project2\results\predictions\lightgbm_application_history_stratified_starter_v1_20261006T080557325548Z_da5c5738.csv
Run metadata: C:\Users\CRNeu\Documents\Data4401\Data4401-Project2\results\predictions\lightgbm_application_history_stratified_starter_v1_20261006T080557325548Z_da5c5738.metadata.json
CSV: 4.06 MiB; metadata: 17.1 KiB
Rows: 46,127; feature set: application_history
Validation-only predictions exported; no model refitted.
